# Paper 1 — Notebook 4 of 4
## Figures, Full 2000–2026 Reconstruction & Zenodo Data Descriptor

**Run order:** NB1 → NB2 → NB3 → **NB4**. Run the first three notebooks first — this one reads all their
outputs from `/kaggle/working/`.

**What NB4 does**
1. Renders all publication figures **F1–F11** to `/kaggle/working/figures/` at 300 dpi
2. Trains the final production correction models on **all mainland stations** (no hold-out) and applies
   them to the **entire ERA5-Land record 2000-07-30 → 2026**, producing the reconstructed daily dataset — *step 1.10*
3. Writes the open CSV dataset + a machine-readable data descriptor for **Zenodo** — *step 1.11*
4. (Optional) Correlates district-mean corrected Tmax with rice yield as an independent signal

**Outputs:** `figures/F1..F11.png`, `bd_agroclim_reconstructed_2000_2026.csv` (+ parquet),
`data_descriptor.json`, `README_dataset.md`, `reconstruction_summary.csv`


### Cell 1 — Imports, style, reload every prior output

In [1]:
import os, json, warnings, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

# Output directory for saving figures and new files
OUT = next((c for c in ['/kaggle/working/','./out/'] if os.path.exists(c)),'./out/')
FIG = OUT + 'figures/'; os.makedirs(FIG, exist_ok=True)

plt.rcParams.update({'figure.dpi':110,'savefig.dpi':300,'font.size':10,
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,
    'grid.alpha':0.25,'axes.axisbelow':True,'figure.facecolor':'white','savefig.bbox':'tight'})
C={'era5':'#2c6fbb','power':'#d1495b','obs':'#333333','corr':'#2a9d8f',
   'interior':'#457b9d','coastal':'#e76f51','hill':'#8a5a44'}

# Define input paths based on your Kaggle directories
PATH_WEATHER_DATASET = '/kaggle/input/datasets/neloypramanik4444/weather-dataset/dataset/'
PATH_V1_1 = '/kaggle/input/notebooks/neloypramanik4444/weatherdata-paper-v1-1/'
PATH_V1_2 = '/kaggle/input/notebooks/neloypramanik4444/weatherdata-paper-v1-2/'
PATH_V1_3 = '/kaggle/input/datasets/neloypramanik4444/weatherdata-paper-v1-3-result-data/'

# Load prior outputs from their respective explicit paths
paired    = pd.read_parquet(PATH_V1_1 + 'paired_era5.parquet')
meta      = pd.read_csv(PATH_V1_1 + 'meta_with_cells.csv')
skill     = pd.read_csv(PATH_WEATHER_DATASET + 'C6_Terrain (slope, TPI, TWI)/terrain_skill_merged_v2.csv')
ter       = pd.read_csv(PATH_WEATHER_DATASET + 'C6_Terrain (slope, TPI, TWI)/terrain_full.csv')
T3        = pd.read_csv(PATH_V1_1 + 'T3_pooled_validation.csv')

# Handle T4 full vs partial based on which exists (checking V1-2 first, falling back to V1-1)
t4_full_path = PATH_V1_2 + 'T4_rainfall_detection_full.csv'
t4_path = PATH_V1_1 + 'T4_rainfall_detection.csv'
T4full = pd.read_csv(t4_full_path) if os.path.exists(t4_full_path) else pd.read_csv(t4_path)

try:    
    land = pd.read_csv(PATH_V1_2 + 'station_landscape.csv')
except Exception:
    land = skill[['station_name']].assign(landscape='Interior plain')
    
try:    
    T9 = pd.read_csv(PATH_V1_3 + 'T9_perstation_before_after.csv')
except Exception: 
    T9 = None
    
try:    
    T8 = pd.read_csv(PATH_V1_3 + 'T8_shap_importance.csv')
except Exception: 
    T8 = None
    
try:    
    island = pd.read_csv(PATH_V1_3 + 'island_holdout_metrics.csv')
except Exception: 
    island = None

print('loaded prior outputs; figures ->', FIG)

loaded prior outputs; figures -> /kaggle/working/figures/


### Cell 2 — F1 Study area (stations on an elevation background) & F2 data availability
F1 places all 36 stations by lat/lon, sized/coloured by elevation, with focal-fill islands marked.
F2 shows per-station completeness as a heat-strip across 2000-2026.

In [2]:
# F1
fig,ax=plt.subplots(figsize=(7,8))
sc=ax.scatter(meta.lon, meta.lat, c=meta.elev_m, s=80, cmap='terrain',
              edgecolor='k', linewidth=0.6, zorder=3)
foc=meta[meta.station_name.isin(['Kutubdia','Sandwip'])]
ax.scatter(foc.lon, foc.lat, s=200, facecolors='none', edgecolors=C['power'],
           linewidth=2, zorder=4, label='focal-filled island')
for _,r in meta.iterrows():
    ax.annotate(r.station_name, (r.lon,r.lat), fontsize=6, xytext=(3,3),
                textcoords='offset points')
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title('F1 — 36 BMD stations over Bangladesh')
plt.colorbar(sc,ax=ax,shrink=0.6,label='Station elevation (m)')
ax.legend(loc='lower left', fontsize=8)
fig.savefig(FIG+'F1_study_area.png'); plt.close(fig)

# F2 availability strip
span=pd.date_range('2000-01-01','2023-12-31',freq='MS')
order=meta.sort_values('lat').station_name.tolist()
avail=np.full((len(order),len(span)),np.nan)
g=paired.assign(ym=paired.date.values.astype('datetime64[M]'))
cov=g.groupby(['station_name','ym']).size().reset_index(name='n')
ymidx={pd.Timestamp(d):i for i,d in enumerate(span)}
sidx={s:i for i,s in enumerate(order)}
for _,r in cov.iterrows():
    if r.station_name in sidx and pd.Timestamp(r.ym) in ymidx:
        avail[sidx[r.station_name], ymidx[pd.Timestamp(r.ym)]]=r.n
fig,ax=plt.subplots(figsize=(11,8))
im=ax.imshow(avail, aspect='auto', cmap='YlGnBu',
             extent=[2000, 2024, len(order), 0], interpolation='nearest')
ax.set_yticks(np.arange(len(order))+0.5); ax.set_yticklabels(order, fontsize=6)
ax.set_xlabel('Year'); ax.set_title('F2 — Data availability (paired station–ERA5 days per month)')
plt.colorbar(im,ax=ax,shrink=0.6,label='days/month')
fig.savefig(FIG+'F2_availability.png'); plt.close(fig)
print('F1, F2 done')

F1, F2 done


### Cell 3 — F3 Taylor diagram (POWER vs ERA5-Land) for Tmax/Tmin/precip
A Taylor diagram summarises correlation, standard-deviation ratio, and centred RMSE in one polar plot.
Built from scratch (no extra package): radius = σ_model/σ_obs, azimuth = arccos(r).

In [3]:
def taylor(ax, stats_list, title):
    # stats_list: list of dicts {label, r, sd_ratio, color, marker}
    ax.set_theta_zero_location('E'); ax.set_theta_direction(1)
    ax.set_thetamin(0); ax.set_thetamax(90)
    rmax=max(1.6, max(s['sd_ratio'] for s in stats_list)*1.15)
    ax.set_ylim(0,rmax)
    # reference arc at r=1 (obs sd)
    th=np.linspace(0,np.pi/2,100)
    ax.plot(th, np.ones_like(th), color='k', lw=0.8, ls='--', alpha=0.5)
    ax.scatter([0],[1.0], c='k', s=60, marker='*', zorder=5, label='Observed')
    for s in stats_list:
        ang=np.arccos(np.clip(s['r'],-1,1))
        ax.scatter([ang],[s['sd_ratio']], c=s['color'], s=70, marker=s['marker'],
                   edgecolor='k', linewidth=0.5, zorder=6, label=s['label'])
    # correlation gridlines
    for rr in [0.3,0.5,0.7,0.9,0.95,0.99]:
        a=np.arccos(rr); ax.plot([a,a],[0,rmax], color='grey', lw=0.4, alpha=0.4)
        ax.text(a, rmax*1.02, f'{rr}', fontsize=6, color='grey')
    ax.set_title(title, fontsize=10, pad=14)
    ax.text(np.pi/4, rmax*1.25, 'correlation', fontsize=7, color='grey',
            rotation=-45, ha='center')

def sd_ratio(df, sim, obs):
    d=df[[sim,obs]].dropna()
    return d[sim].std()/d[obs].std()

# FIXED: Changed OUT to PATH_V1_1
power = pd.read_parquet(PATH_V1_1 + 'paired_power.parquet')

# obs column names differ: ERA5 paired uses obs_*, POWER paired uses bare tmax/tmin/prcp
POWER_OBS={'Tmax':'tmax','Tmin':'tmin','Precip':'prcp'}
fig,axes=plt.subplots(1,3, subplot_kw={'projection':'polar'}, figsize=(15,5.2))
panels=[('Tmax','t2m_max','obs_tmax','pw_tmax'),
        ('Tmin','t2m_min','obs_tmin','pw_tmin'),
        ('Precip','era5_prcp','obs_prcp','pw_prcp')]

for ax,(lab,ecol,ocol,pcol) in zip(axes,panels):
    e=T3.query("product=='ERA5-Land' and variable==@lab").iloc[0]
    p=T3.query("product=='NASA POWER' and variable==@lab").iloc[0]
    sl=[dict(label='ERA5-Land', r=e['r'], sd_ratio=sd_ratio(paired,ecol,ocol), color=C['era5'], marker='o'),
        dict(label='NASA POWER', r=p['r'], sd_ratio=sd_ratio(power,pcol,POWER_OBS[lab]), color=C['power'], marker='s')]
    taylor(ax, sl, f'F3 — {lab}')
    
axes[0].legend(loc='upper right', bbox_to_anchor=(0.15,1.12), fontsize=7)
fig.suptitle('F3 — Taylor diagram: ERA5-Land vs NASA POWER', y=1.05)
fig.savefig(FIG+'F3_taylor.png'); plt.close(fig)
print('F3 done')

F3 done


### Cell 4 — ⭐ F4 Terrain controls on bias
The headline figure. Left: `land_frac_9km` vs Tmax bias with OLS fit + 95% CI. Right: `elev_std_9km`
vs tmax_r with fit + CI. Input is the per-station skill table.

In [4]:
from scipy import stats as ss
def scatter_fit(ax, x, y, xlabel, ylabel, title, color, labelpts=None, logx=False):
    x=np.asarray(x,float); y=np.asarray(y,float); m=np.isfinite(x)&np.isfinite(y); x,y=x[m],y[m]
    ax.scatter(x,y,s=55,c=color,edgecolor='k',linewidth=0.5,zorder=3)
    xs=np.linspace(x.min(),x.max(),100)
    sl,ic,r,p,se=ss.linregress(x,y)
    ax.plot(xs, ic+sl*xs, color='k', lw=1.5, zorder=4)
    # CI band
    n=len(x); dof=n-2; tval=ss.t.ppf(0.975,dof)
    xm=x.mean(); ssx=np.sum((x-xm)**2); resid=y-(ic+sl*x); s_err=np.sqrt(np.sum(resid**2)/dof)
    ci=tval*s_err*np.sqrt(1/n+(xs-xm)**2/ssx)
    ax.fill_between(xs,(ic+sl*xs)-ci,(ic+sl*xs)+ci,color=color,alpha=0.15,zorder=2)
    rho,pr=ss.spearmanr(x,y)
    ax.set_xlabel(xlabel);ax.set_ylabel(ylabel)
    ax.set_title(f'{title}\nSpearman ρ={rho:+.2f} (p={pr:.3f}),  R²={r**2:.2f}')
    return rho,pr

fig,axes=plt.subplots(1,2,figsize=(13,5.4))
scatter_fit(axes[0], skill.land_frac_9km, skill.tmax_bias,
            'land_frac_9km (fraction land in 9 km cell)','Tmax bias (°C)',
            'F4a — Land fraction drives cold bias', C['era5'])
scatter_fit(axes[1], skill.elev_std_9km, skill.tmax_r,
            'elev_std_9km (sub-grid elevation SD, m)','Tmax correlation r',
            'F4b — Sub-grid relief degrades correlation', C['coastal'])
# annotate extreme stations
for _,r in skill.iterrows():
    if r.land_frac_9km<0.45 or r.tmax_bias<-2.6:
        axes[0].annotate(r.station_name,(r.land_frac_9km,r.tmax_bias),fontsize=6,xytext=(3,3),textcoords='offset points')
    if r.elev_std_9km>20:
        axes[1].annotate(r.station_name,(r.elev_std_9km,r.tmax_r),fontsize=6,xytext=(3,3),textcoords='offset points')
fig.suptitle('F4 — Terrain controls on ERA5-Land bias (n=36)  ⭐', y=1.02, fontweight='bold')
fig.savefig(FIG+'F4_terrain_controls.png'); plt.close(fig)
print('F4 done  (headline figure)')

F4 done  (headline figure)


### Cell 5 — F5 Seasonal error cycle by landscape type & F6 error-vs-rainfall-bin + POD/FAR curves

In [5]:
d=paired.merge(land,on='station_name',how='left')
d['month']=d.date.dt.month
d['tmax_err']=d.t2m_max-d.obs_tmax

# F5 seasonal cycle by landscape
fig,ax=plt.subplots(figsize=(9,5.2))
cmap_ls={'Interior plain':C['interior'],'Coastal / island':C['coastal'],'Hill-adjacent':C['hill']}
for lt,gg in d.groupby('landscape'):
    mc=gg.groupby('month')['tmax_err'].mean()
    sd=gg.groupby('month')['tmax_err'].std()/np.sqrt(gg.groupby('month').size())
    ax.plot(mc.index,mc.values,marker='o',color=cmap_ls.get(lt,'grey'),label=lt)
    ax.fill_between(mc.index,mc.values-sd.values,mc.values+sd.values,color=cmap_ls.get(lt,'grey'),alpha=0.15)
ax.axhline(0,color='k',lw=0.8)
ax.set_xticks(range(1,13)); ax.set_xlabel('Month'); ax.set_ylabel('Tmax error (model−obs, °C)')
ax.set_title('F5 — Seasonal Tmax error cycle by landscape type'); ax.legend(fontsize=8)
fig.savefig(FIG+'F5_seasonal_error.png'); plt.close(fig)

# F6 error vs observed-rainfall bin + POD/FAR curves
bins=[0,1,5,10,20,50,1000]; labels=['0–1','1–5','5–10','10–20','20–50','>50']
d['rbin']=pd.cut(d.obs_prcp,bins=bins,labels=labels,include_lowest=True)
d['prcp_err']=d.era5_prcp-d.obs_prcp
fig,axes=plt.subplots(1,2,figsize=(13,5.2))
bp=d.groupby('rbin')['prcp_err'].mean()
be=d.groupby('rbin')['prcp_err'].std()/np.sqrt(d.groupby('rbin').size())
axes[0].bar(range(len(bp)),bp.values,yerr=be.values,color=C['era5'],alpha=0.8,capsize=3)
axes[0].set_xticks(range(len(bp))); axes[0].set_xticklabels(bp.index)
axes[0].axhline(0,color='k',lw=0.8)
axes[0].set_xlabel('Observed rainfall bin (mm/day)'); axes[0].set_ylabel('Mean precip error (mm)')
axes[0].set_title('F6a — Precip error grows with rainfall intensity')
# POD/FAR from T4 full
raw=T4full[T4full['stage'].astype(str).str.contains('raw', case=False)].drop_duplicates('threshold')
axes[1].plot(raw.threshold,raw.POD,marker='o',color=C['corr'],label='POD')
axes[1].plot(raw.threshold,raw.FAR,marker='s',color=C['power'],label='FAR')
axes[1].plot(raw.threshold,raw.CSI,marker='^',color=C['era5'],label='CSI')
axes[1].set_xscale('log'); axes[1].set_xticks([1,10,20,50]); axes[1].get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
axes[1].set_xlabel('Threshold (mm/day)'); axes[1].set_ylabel('Score')
axes[1].set_title('F6b — Detection skill vs threshold (POD collapse)'); axes[1].legend(fontsize=8)
fig.savefig(FIG+'F6_rainfall_error.png'); plt.close(fig)
print('F5, F6 done')

F5, F6 done


### Cell 6 — F7 Spatial error maps (4 panels): Tmax bias, Tmax r, Tmin r, precip r

In [6]:
fig,axes=plt.subplots(2,2,figsize=(12,13))
sk=skill.merge(meta[['station_name','lat','lon']],on='station_name',how='left')
panels=[('tmax_bias','Tmax bias (°C)','coolwarm_r',axes[0,0]),
        ('tmax_r','Tmax r','viridis',axes[0,1]),
        ('tmin_r','Tmin r','viridis',axes[1,0]),
        ('prcp_r','Precip r','magma',axes[1,1])]
for col,lab,cm,ax in panels:
    sc=ax.scatter(sk.lon,sk.lat,c=sk[col],s=110,cmap=cm,edgecolor='k',linewidth=0.5)
    ax.set_xlabel('Lon');ax.set_ylabel('Lat');ax.set_title(f'F7 — {lab}')
    plt.colorbar(sc,ax=ax,shrink=0.7,label=lab)
fig.suptitle('F7 — Spatial pattern of ERA5-Land skill',y=1.0)
fig.savefig(FIG+'F7_spatial_error.png'); plt.close(fig)
print('F7 done')

F7 done


### Cell 7 — F8 LOSO RMSE box plots across methods & F9 SHAP beeswarm

In [7]:
# F8 — build from T7 if present, else from per-station T9
fig,ax=plt.subplots(figsize=(9,5.2))
if os.path.exists(OUT+'T7_ml_model_comparison.csv'):
    T7=pd.read_csv(OUT+'T7_ml_model_comparison.csv')
    # keep Tmax rows: RAW baseline + LOSO with-neighbour models (one row per method)
    tm=T7[(T7.variable=='Tmax') & (T7.features.isin(['-','with_nb']))].copy()
    tm=tm.drop_duplicates(subset='method', keep='first').sort_values('rmse')
    ax.bar(range(len(tm)), tm['rmse'].values,
           color=[C['power'] if 'RAW' in m else C['corr'] for m in tm['method']])
    ax.set_xticks(range(len(tm))); ax.set_xticklabels(tm['method'],rotation=30,ha='right')
    ax.set_ylabel('Tmax RMSE (°C), LOSO'); ax.set_title('F8 — LOSO RMSE across correction methods (Tmax)')
else:
    ax.text(0.5,0.5,'Run NB3 for T7',ha='center')
fig.savefig(FIG+'F8_loso_rmse.png'); plt.close(fig)

# F9 SHAP beeswarm (if raw shap persisted) else importance bar
fig,ax=plt.subplots(figsize=(8,6))
if os.path.exists(OUT+'shap_values_tmax.parquet'):
    sv=pd.read_parquet(OUT+'shap_values_tmax.parquet')
    xs=pd.read_parquet(OUT+'shap_features_tmax.parquet') if os.path.exists(OUT+'shap_features_tmax.parquet') else None
    order=sv.abs().mean().sort_values(ascending=False).index[:12][::-1]
    for i,f in enumerate(order):
        vals=sv[f].values
        jitter=(np.random.rand(len(vals))-0.5)*0.6
        if xs is not None:
            fv=xs[f].values; fn=(fv-np.nanpercentile(fv,5))/(np.nanpercentile(fv,95)-np.nanpercentile(fv,5)+1e-9)
            ax.scatter(vals, i+jitter, c=np.clip(fn,0,1), cmap='coolwarm', s=6, alpha=0.5)
        else:
            ax.scatter(vals, i+jitter, s=6, alpha=0.4, color=C['era5'])
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order,fontsize=8)
    ax.axvline(0,color='k',lw=0.8)
    ax.set_xlabel('SHAP value (impact on Tmax residual)')
    ax.set_title('F9 — SHAP beeswarm (Tmax correction model)')
elif T8 is not None:
    top=T8.head(12)[::-1]
    ax.barh(range(len(top)),top.mean_abs_shap,color=C['era5'])
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top.feature,fontsize=8)
    ax.set_xlabel('mean |importance|'); ax.set_title('F9 — Feature importance (Tmax)')
fig.savefig(FIG+'F9_shap.png'); plt.close(fig)
print('F8, F9 done')

F8, F9 done


### Cell 8 — F10 Before/after series (one interior + one coastal) & F11 island focal-fill diagnostic

In [8]:
# choose an interior (Ishurdi) and coastal (Cox's Bazar) example
def monthly(df,col): 
    return df.set_index('date')[col].resample('MS').mean()
examples=[('Ishurdi','interior'),("Cox's Bazar",'coastal')]
fig,axes=plt.subplots(2,1,figsize=(12,8),sharex=False)
for ax,(stn,kind) in zip(axes,examples):
    g=paired[paired.station_name==stn].sort_values('date')
    om=monthly(g,'obs_tmax'); em=monthly(g,'t2m_max')
    ax.plot(om.index,om.values,color=C['obs'],lw=1,label='Observed')
    ax.plot(em.index,em.values,color=C['era5'],lw=1,alpha=0.8,label='ERA5-Land raw')
    # simple LS-corrected preview (mean shift on overlap) to visualise
    shift=(om-em).mean(); ax.plot(em.index,(em+shift).values,color=C['corr'],lw=1,alpha=0.9,label='ERA5-Land + LS')
    ax.set_title(f'F10 — {stn} ({kind}) monthly Tmax'); ax.set_ylabel('Tmax (°C)'); ax.legend(fontsize=8)
fig.suptitle('F10 — Before/after Tmax at an interior and a coastal station',y=1.0)
fig.savefig(FIG+'F10_before_after.png'); plt.close(fig)

# F11 island diagnostic
fig,ax=plt.subplots(figsize=(9,5.2))
if island is not None and 'raw_bias' in island.columns:
    x=np.arange(len(island)); w=0.35
    ax.bar(x-w/2, island.raw_rmse, w, color=C['power'], label='raw RMSE')
    ax.bar(x+w/2, island.corr_rmse, w, color=C['corr'], label='corrected RMSE')
    ax.set_xticks(x); ax.set_xticklabels(island.station, fontsize=9)
    ax.set_ylabel('Tmax RMSE (°C)')
    for i,r in island.iterrows():
        ax.annotate(f'bias {r.raw_bias:+.2f}→{r.corr_bias:+.2f}',(i,max(r.raw_rmse,r.corr_rmse)+0.05),
                    ha='center',fontsize=7)
    ax.set_title('F11 — Island focal-fill diagnostic: Kutubdia & Sandwip'); ax.legend(fontsize=8)
else:
    ax.text(0.5,0.5,'Run NB3 for island metrics',ha='center')
fig.savefig(FIG+'F11_island.png'); plt.close(fig)
print('F10, F11 done')
print('\nAll figures written to', FIG)
print(sorted(os.listdir(FIG)))

F10, F11 done

All figures written to /kaggle/working/figures/
['F10_before_after.png', 'F11_island.png', 'F1_study_area.png', 'F2_availability.png', 'F3_taylor.png', 'F4_terrain_controls.png', 'F5_seasonal_error.png', 'F6_rainfall_error.png', 'F7_spatial_error.png', 'F8_loso_rmse.png', 'F9_shap.png']


### Cell 9 — Final production models & full 2000–2026 reconstruction — *step 1.10*
Train the chosen correction on **all mainland stations** (no LOSO hold-out — this is the deployment model),
then apply to the entire ERA5-Land record for every station, including the post-2023 period that has no
station data to validate against. Focal-filled islands are corrected with the mainland model and flagged.

In [9]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import clone

# rebuild features on the FULL ERA5 record (not just the paired/overlap window)
# FIXED: Replaced BASE with PATH_WEATHER_DATASET and added the correct subfolder
era_full=pd.read_csv(PATH_WEATHER_DATASET + 'era5land_stations_daily/era5land_stations_daily_v2.csv', parse_dates=['date'])
era_full=era_full.merge(ter[['station_name','land_frac_9km','elev_std_9km','roughness_5km',
                             'dist_water_km','elev_dem','northness','eastness','tpi_5km']],
                        on='station_name', how='left')
era_full['doy']=era_full.date.dt.dayofyear
era_full['doy_sin']=np.sin(2*np.pi*era_full.doy/365.25)
era_full['doy_cos']=np.cos(2*np.pi*era_full.doy/365.25)
era_full['month']=era_full.date.dt.month

# neighbour features on the full record
def hav(a1,o1,a2,o2):
    R=6371.0;p1,p2=np.radians(a1),np.radians(a2);dphi=np.radians(a2-a1);dl=np.radians(o2-o1)
    x=np.sin(dphi/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2;return 2*R*np.arcsin(np.sqrt(x))
coords=meta.set_index('station_name')[['lat','lon']]; stations=list(coords.index)
Dm=pd.DataFrame(index=stations,columns=stations,dtype=float)
for a in stations:
    for b in stations: Dm.loc[a,b]=0.0 if a==b else hav(*coords.loc[a],*coords.loc[b])
neighbours={s:[x for x in Dm.loc[s].sort_values().index if x!=s][:3] for s in stations}
def nf_full(df,val):
    piv=df.pivot_table(index='date',columns='station_name',values=val,aggfunc='mean')
    tbl=pd.DataFrame({s:piv[neighbours[s]].mean(axis=1) for s in stations})
    long=tbl.reset_index().melt(id_vars='date',var_name='station_name',value_name='nbv')
    return df.merge(long,on=['date','station_name'],how='left')['nbv'].values
era_full=era_full.rename(columns={'t2m_max':'t2m_max','t2m_min':'t2m_min','prcp':'era5_prcp'})
era_full['nb_tmax']=nf_full(era_full,'t2m_max')
era_full['nb_tmin']=nf_full(era_full,'t2m_min')
era_full['nb_prcp']=nf_full(era_full,'era5_prcp')

FEATS=['t2m_max','t2m_min','era5_prcp','ssrd','ws2','rh_mean','vpd','swvl1','sp',
       'land_frac_9km','elev_std_9km','roughness_5km','dist_water_km','elev_dem',
       'northness','eastness','tpi_5km','doy_sin','doy_cos','month','nb_tmax','nb_tmin','nb_prcp']

# training data = paired overlap on mainland stations
FOCAL_FILLED=['Kutubdia','Sandwip']
train=paired.merge(ter[['station_name','land_frac_9km','elev_std_9km','roughness_5km',
                        'dist_water_km','elev_dem','northness','eastness','tpi_5km']],
                   on='station_name',how='left')
train['doy']=train.date.dt.dayofyear
train['doy_sin']=np.sin(2*np.pi*train.doy/365.25); train['doy_cos']=np.cos(2*np.pi*train.doy/365.25)
train['month']=train.date.dt.month
train['nb_tmax']=nf_full(train.rename(columns={}),'t2m_max')  # neighbours within paired window
train['nb_tmin']=nf_full(train,'t2m_min'); train['nb_prcp']=nf_full(train,'era5_prcp')
train=train[train.station_name.isin([s for s in stations if s not in FOCAL_FILLED])]

def fit_apply(target_obs, mod_col, tag):
    tr=train.copy(); tr['res']=tr[target_obs]-tr[mod_col]
    trc=tr.dropna(subset=FEATS+['res'])
    if len(trc)>200000: trc=trc.sample(200000,random_state=42)
    mdl=ExtraTreesRegressor(n_estimators=200,max_depth=25,min_samples_leaf=5,
                            n_jobs=4,random_state=42).fit(trc[FEATS].values, trc['res'].values)
    ff=era_full.dropna(subset=FEATS).copy()
    ff[tag]=ff[mod_col].values + mdl.predict(ff[FEATS].values)
    return ff[['station_name','date',tag]]

corr_tmax=fit_apply('obs_tmax','t2m_max','tmax_corr')
corr_tmin=fit_apply('obs_tmin','t2m_min','tmin_corr')
corr_prcp=fit_apply('obs_prcp','era5_prcp','prcp_corr')
print('reconstruction predictions computed for tmax/tmin/prcp')

reconstruction predictions computed for tmax/tmin/prcp


### Cell 10 — Assemble the open reconstructed dataset (2000–2026) + provenance flags

In [10]:
recon=(era_full[['station_name','date','t2m_max','t2m_min','era5_prcp','era5_source']]
       .merge(corr_tmax,on=['station_name','date'],how='left')
       .merge(corr_tmin,on=['station_name','date'],how='left')
       .merge(corr_prcp,on=['station_name','date'],how='left'))
recon=recon.rename(columns={'t2m_max':'tmax_era5_raw','t2m_min':'tmin_era5_raw','era5_prcp':'prcp_era5_raw'})
# non-negative precip after correction
recon['prcp_corr']=recon['prcp_corr'].clip(lower=0)
recon['focal_filled']=recon.station_name.isin(['Kutubdia','Sandwip'])
recon['in_validation_period']=recon.date<='2023-12-31'
recon['partial_year_2026']=recon.date.dt.year==2026
recon=recon.merge(meta[['station_name','lat','lon','elev_m','division','district']],on='station_name',how='left')
recon=recon.sort_values(['station_name','date']).reset_index(drop=True)

csv_path=OUT+'bd_agroclim_reconstructed_2000_2026.csv'
recon.to_csv(csv_path,index=False)
recon.to_parquet(OUT+'bd_agroclim_reconstructed_2000_2026.parquet',index=False)
print('reconstructed dataset rows:',len(recon),'->',csv_path)

# summary table
summ=(recon.groupby('station_name')
      .agg(n_days=('date','size'),
           start=('date','min'), end=('date','max'),
           mean_tmax_raw=('tmax_era5_raw','mean'),
           mean_tmax_corr=('tmax_corr','mean'),
           focal=('focal_filled','first')).reset_index())
summ['mean_tmax_shift']=(summ.mean_tmax_corr-summ.mean_tmax_raw).round(2)
summ.to_csv(OUT+'reconstruction_summary.csv',index=False)
print(summ[['station_name','n_days','mean_tmax_raw','mean_tmax_corr','mean_tmax_shift','focal']].head(10).to_string(index=False))

reconstructed dataset rows: 348756 -> /kaggle/working/bd_agroclim_reconstructed_2000_2026.csv
 station_name  n_days  mean_tmax_raw  mean_tmax_corr  mean_tmax_shift  focal
Ambagan (Ctg)    9708      28.676101       31.246072             2.57  False
      Barisal    9708      29.512213       31.068568             1.56  False
        Bhola    9708      29.207159       30.842385             1.64  False
        Bogra    9708      29.605730       30.869689             1.26  False
     Chandpur    9708      29.493841       31.048615             1.55  False
   Chittagong    9708      28.396301       30.640832             2.24  False
    Chuadanga    9708      30.259204       31.635013             1.38  False
      Comilla    9708      29.704071       30.658247             0.95  False
  Cox's Bazar    9708      28.492358       31.199207             2.71  False
        Dhaka    9708      29.741119       31.054909             1.31  False


### Cell 11 — Zenodo data descriptor (JSON) + dataset README — *step 1.11*

In [11]:
descriptor={
 "title":"BD-AgroClim: ML-reconstructed daily agro-climatic dataset for Bangladesh (2000-2026)",
 "version":"1.0",
 "temporal_coverage":{"start":str(recon.date.min().date()),"end":str(recon.date.max().date()),
                      "validation_period":"2000-01-01/2023-12-31","note":"2026 is Jan-Jul partial"},
 "spatial_coverage":{"stations":int(recon.station_name.nunique()),"country":"Bangladesh",
                     "extraction":"ERA5-Land 0.1deg at BMD station coordinates"},
 "variables":{
   "tmax_corr":"ML bias-corrected daily max 2 m temperature (°C)",
   "tmin_corr":"ML bias-corrected daily min 2 m temperature (°C)",
   "prcp_corr":"ML bias-corrected daily precipitation (mm), non-negative",
   "*_era5_raw":"uncorrected ERA5-Land source values",
   "focal_filled":"true for Kutubdia & Sandwip (interpolated coastal cells, not native ERA5-Land)",
   "in_validation_period":"true where BMD station data exists for validation"},
 "methods":{"correction":"Extra Trees residual learning on reanalysis + terrain + calendar + neighbouring-station features",
            "validation":"Leave-One-Station-Out cross-validation",
            "key_predictors":["land_frac_9km","elev_std_9km","swvl1","vpd","roughness_5km"],
            "soil_moisture_note":"GWETTOP/GWETROOT (POWER) are dimensionless wetness fractions 0-1, not volumetric"},
 "provenance":{"reanalysis":["NASA POWER 0.5deg","ERA5-Land 0.1deg (ECMWF/ERA5_LAND/DAILY_AGGR)"],
               "terrain":["Copernicus GLO-30","SRTM v3","JRC Global Surface Water"],
               "stations":"Bangladesh Meteorological Department (36 long-record)"},
 "known_limitations":["daily precipitation detection skill limited at heavy thresholds (POD low at >=50 mm)",
                      "islands corrected with mainland model (see focal_filled)",
                      "post-2023 values are unvalidated extrapolation"],
 "license":"CC-BY-4.0",
 "files":["bd_agroclim_reconstructed_2000_2026.csv","reconstruction_summary.csv",
          "T1_station_inventory.csv","T3_pooled_validation.csv","T7_ml_model_comparison.csv"]
}
with open(OUT+'data_descriptor.json','w') as f: json.dump(descriptor,f,indent=2)

readme=f'''# BD-AgroClim Reconstructed Dataset (2000-2026)

ML bias-corrected daily Tmax, Tmin and precipitation at 36 Bangladesh Meteorological
Department stations, derived from ERA5-Land and benchmarked against NASA POWER.

**Rows:** {len(recon):,}  |  **Stations:** {recon.station_name.nunique()}  |  **Period:** {recon.date.min().date()} to {recon.date.max().date()}

## Columns
- `station_name, date, lat, lon, elev_m, division, district`
- `tmax_corr, tmin_corr, prcp_corr` — corrected values (use these)
- `tmax_era5_raw, tmin_era5_raw, prcp_era5_raw` — uncorrected ERA5-Land
- `era5_source` — native / focal_filled
- `focal_filled` — Kutubdia & Sandwip flag
- `in_validation_period` — true up to 2023-12-31
- `partial_year_2026` — exclude from annual statistics

## Method
Extra Trees residual learning (obs − ERA5-Land) using reanalysis + terrain
(`land_frac_9km`, `elev_std_9km`, roughness, distance-to-water) + calendar +
neighbouring-station features. Validated by Leave-One-Station-Out CV.

## Caveats
Post-2023 values are unvalidated extrapolation. Daily heavy-rain detection remains
limited (POD low at >=50 mm/day). Islands are corrected with the mainland model.

License: CC-BY-4.0
'''
open(OUT+'README_dataset.md','w').write(readme)
print('Wrote data_descriptor.json and README_dataset.md')
print('\nNB4 complete. Paper 1 pipeline finished: tables T1-T9, figures F1-F11, reconstructed dataset + descriptor.')

Wrote data_descriptor.json and README_dataset.md

NB4 complete. Paper 1 pipeline finished: tables T1-T9, figures F1-F11, reconstructed dataset + descriptor.


### Cell 12 — (Optional) Yield cross-signal
A lightweight independent check: does district-mean corrected Tmax relate to rice yield? This is a
Paper-4 topic; here it is only a sanity signal that the corrected field carries agronomic information.

In [12]:
try:
    # FIXED: Replaced BASE with PATH_WEATHER_DATASET and added the correct subfolder
    y = pd.read_csv(PATH_WEATHER_DATASET + 'BBS Crop Yield/yield_panel_final.csv').query('analysis_ready == True')
    
    # crude district match by name; annual mean corrected Tmax
    recon['year'] = recon.date.dt.year
    dt = recon.groupby(['district','year'])['tmax_corr'].mean().reset_index()
    y['year'] = y['harvest_year'] if 'harvest_year' in y else y['year_start']
    m = y.merge(dt, on=['district','year'], how='inner')
    
    if len(m) > 30:
        from scipy import stats as ss
        r, p = ss.spearmanr(m['tmax_corr'], m['yield_kg_ha'])
        print(f'District-year corrected Tmax vs rice yield: Spearman r={r:+.3f} (p={p:.3g}, n={len(m)})')
        print('Interpretation: exploratory only; full attribution is Paper 4.')
    else:
        print('Insufficient district-name overlap for a robust signal (expected; Paper 4 handles mapping).')
except Exception as e:
    print('Yield cross-signal skipped:', e)

District-year corrected Tmax vs rice yield: Spearman r=+0.042 (p=0.14, n=1235)
Interpretation: exploratory only; full attribution is Paper 4.
